# Liquidity Absorption 04 — What happens between entry and exit?

**Outcome-conditioned intratrade anatomy, not a new alpha signal or validated explanation.**

Primary cohort: A03d `VD18_VOL_FIXED`, **240-minute** taker entry, the same 452 A02 candidate IDs. Compare TARGET vs STOP vs TIMEOUT, but only among positions still open at exact causal **1, 5, 15, 30 and 60-minute landmarks** after the completed signal candle. Minute features use only fully closed one-minute UM futures klines, with taker-buy quote volume as an *aggressive flow proxy*. No future market information enters the feature vector. Subsequent outcome is joined for *descriptive comparison only*.

Do **not** equate any winner/loss feature difference with a causal reason or a deployable filter. Target distance, sigma and stop geometry may explain much of the separation; only later held-out data can support prediction. No new tick archives; existing `BTCUSDT_um_1m.parquet` / `ETHUSDT_um_1m.parquet` are read once each.


In [1]:
from pathlib import Path
import sys, subprocess
import pandas as pd
from IPython.display import display, Image
REPO=Path.cwd().resolve()
while REPO!=REPO.parent and not (REPO/'src'/'data_engineering').is_dir():
    REPO=REPO.parent
assert (REPO/'src'/'data_engineering').is_dir(), 'Open Jupyter within repo'
if str(REPO) not in sys.path:sys.path.insert(0,str(REPO))
from src.data_engineering.modeling.liquidity_absorption_stage_a import binance_root
from src.data_engineering.modeling.liquidity_absorption_stage_a04 import VERSION,Config,run
assert VERSION=='LA-A04-0.1.0', 'git pull and restart kernel'
print('A04',VERSION)
r=subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-p','test_liquidity_absorption_stage_a04.py','-q'],cwd=REPO,capture_output=True,text=True)
print(r.stdout[-3000:],r.stderr[-3000:]);assert r.returncode==0, 'A04 synthetic regression tests failed'
FUTURES_ROOT=binance_root(REPO)
OUT=REPO/'notebooks'/'Liquidity_Absorption'/'04'
print('Futures archive',FUTURES_ROOT)


A04 LA-A04-0.1.0
 ----------------------------------------------------------------------
Ran 14 tests in 0.123s

OK

Futures archive D:\MarketData\Binance


In [2]:
report=run(REPO,FUTURES_ROOT,OUT,Config())
display(report)


ValueError: Index has duplicate keys: Index([1787740, 1787741, 1787742, 1787743, 1787744, 1787745, 1787746, 1787747,
       1787748, 1787749,
       ...
       1851754, 1851755, 1851756, 1851757, 1851758, 1851759, 1851760, 1851761,
       1851762, 1851763],
      dtype='int64', name='close_ms', length=64024)

## Outcome cohort and censoring

First examine TARGET, STOP, TIMEOUT, skipped and censored events separately. The trade-entry/exit labels come from the *frozen A03d* 240-minute risk policy, not a re-optimized strategy. The cumulative curves retain the 452-event denominator, and censored trades are not invented wins or losses.


In [ ]:
events=pd.read_csv(OUT/'04_primary_event_outcomes.csv')
exits=pd.read_csv(OUT/'04_competing_exits.csv')
print('Primary 240m outcome distribution:')
display(events.groupby(['symbol','reason']).size().rename('n').reset_index())
display(exits.query('symbol=="ALL"'))
display(Image(filename=str(OUT/'04_competing_exits.png')))


## At-risk intratrade landmarks

Features at +1/+5/+15/+30/+60 minutes use only one-minute candles *closed* by that checkpoint. A position is represented at a checkpoint only if its A03d exit occurs **strictly after** it. Show counts dropped due to earlier exits or missing minutes before discussing medians.

- **Price acceptance**: directional close return; fraction of completed bars closing on favorable side of liquidity level; early zone-retests; maximum favorable/adverse candle excursions
- **Flow**: directional taker-buy quote imbalance and its opposing-aggressor complement
- **Activity**: post-signal per-minute quote volume relative to 30 minutes known at signal
- **Geometry**: progress toward the frozen target, stop-distance R, and target distance relative to the frozen 60-minute forecast sigma

High/low-based excursions are retrospective within the completed minutes; they cannot prove the ordering of prints *within the same minute*. Aggressor imbalance is **not** order-book absorption or passive queue data.


In [ ]:
panel=pd.read_csv(OUT/'04_landmark_panel.csv')
summary=pd.read_csv(OUT/'04_landmark_summary.csv')
print('Risk-set accounting:')
display(panel.groupby(['landmark_min','observation_status']).size().rename('n').unstack(fill_value=0))
print('Causal checkpoint feature summaries by BTC/ETH and eventual outcome:')
cols=['symbol','landmark_min','eventual_outcome','n_at_risk','median_signed_return_bp','median_max_adverse_bp','median_flow_alignment','median_quote_activity_vs_past30','median_level_favorable_close_share','median_zone_overlap_minute_fraction','median_target_over_sigma60']
display(summary[cols].round(3))
display(Image(filename=str(OUT/'04_landmark_features.png')))


## Research interpretation / next decision

Differences visible at +5m or +15m may motivate frozen continuation features. They are **not** an independently tested signal. Keep symbol, direction, structural target type and target/sigma ratio visible as confounders. Outcomes depend heavily on barrier distance and the 240m time limit. Use time-blocked prospective evaluation **after** freezing any rule. Historical July–September 2026 is already inspected. This notebook does not edit A02/A03/VD18 or collectors and never requires tick files.
